# DANTE-IR: audit and interpretable analysis of infrared spectra of biofluids

**What it does.** DANTE-IR checks whether the spectra of your two groups (for example, patients and controls) differ because of the **samples** or because of **how they were measured** (instrument, environment, session, collection site), and describes the differences with interpretable band variables (proteins, lipids, phosphates, carbohydrates, hydration).

**How to use it** (no programming needed): run the four cells below **in order**, clicking the ▶ button of each one, and fill in the form of step 3. At the end, a ZIP file with the report, figures and tables is downloaded.

**Your data** is processed only in your own Colab session; it is not sent anywhere else.

**Accepted files**
- *One row per spectrum* (recommended): a CSV or Excel table whose columns are the wavenumbers (cm⁻¹) plus a column with the group of each spectrum and, if there are several spectra per person, a column identifying the person.
- *One row per wavenumber*: the first column holds the wavenumbers and each other column is a spectrum; then also upload a metadata table with one row per spectrum (first column: the spectrum name; other columns: group, person, batch...).
- Spectra must cover at least 900–3100 cm⁻¹ (ideally up to 4000 cm⁻¹). Absorbance or transmittance (%) are both accepted.

In [ ]:
#@title 1. Install DANTE-IR (once per session)
#@markdown Leave the address as it is. If you received a `.whl` file instead, clear the box and you will be asked to upload it.
PACKAGE_SOURCE = "https://github.com/YOUR_USER/dante-ir/archive/refs/heads/main.zip" #@param {type:"string"}
import subprocess, sys
if PACKAGE_SOURCE.strip() and "YOUR_USER" not in PACKAGE_SOURCE:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", PACKAGE_SOURCE.strip()])
else:
    from google.colab import files
    print("Upload the DANTE-IR .whl file")
    _up = files.upload()
    _whl = [f for f in _up if f.endswith(".whl")]
    if not _whl:
        raise SystemExit("No .whl file was uploaded.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _whl[0]])
import dante_ir
print("DANTE-IR", dante_ir.__version__, "is ready.")

In [ ]:
#@title 2. Upload your data
#@markdown Select the spectra file and, if needed, the metadata file. The columns of each file are listed below to help you fill in step 3.
from google.colab import files
from dante_ir.io import read_table
_uploaded = list(files.upload().keys())
for _f in _uploaded:
    try:
        _t = read_table(_f)
        _non_numeric = [c for c in _t.columns if not str(c).replace(".", "", 1).replace("-", "", 1).isdigit()]
        print(f"\n{_f}: {_t.shape[0]} rows x {_t.shape[1]} columns")
        print("  Non-numeric columns (candidates for class / person / batch):", _non_numeric[:30])
        for _c in _non_numeric[:10]:
            _v = _t[_c].dropna().astype(str).unique()
            if len(_v) <= 10:
                print(f"    {_c}: {list(_v)}")
    except Exception as _e:
        print(f"{_f}: could not be read ({_e})")

In [ ]:
#@title 3. Describe your data, then run this cell
spectra_file = "" #@param {type:"string"}
#@markdown Leave empty if the spectra file already contains the class column:
metadata_file = "" #@param {type:"string"}
#@markdown Column with the group of each spectrum and the value that identifies the disease / positive group:
class_column = "label" #@param {type:"string"}
positive_class = "" #@param {type:"string"}
#@markdown Column identifying the person, if there are several spectra per person (strongly recommended):
subject_column = "" #@param {type:"string"}
#@markdown Optional column with the batch, session, day or collection site (enables the batch audit):
batch_column = "" #@param {type:"string"}
sample_type = "saliva" #@param ["saliva", "serum", "plasma", "blood", "urine", "other"]
atr_crystal = "diamond" #@param ["diamond", "ZnSe", "Ge", "Si", "transmission"]
analysis = "Audit (minutes)" #@param ["Audit (minutes)", "Exploratory (about 1 hour)", "Full (several hours)"]
dataset_name = "my_dataset" #@param {type:"string"}
#@markdown Save the results in Google Drive (recommended for long analyses):
save_to_drive = False #@param {type:"boolean"}

if not spectra_file and "_uploaded" in globals() and _uploaded:
    spectra_file = _uploaded[0]
if not positive_class:
    raise SystemExit("Please write the positive class (the value of the class column for the disease group).")
print("Settings saved. Now run step 4.")

In [ ]:
#@title 4. Run the analysis
import os
from google.colab import files
_mode = {"Audit (minutes)": "audit", "Exploratory (about 1 hour)": "exploratory",
         "Full (several hours)": "full"}[analysis]
if save_to_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    _out = f"/content/drive/MyDrive/DANTE-IR/{dataset_name}_{_mode}"
else:
    _out = f"/content/{dataset_name}_{_mode}"
result = dante_ir.run(spectra_file, class_column, positive_class,
                      metadata=metadata_file or None, subject_column=subject_column or None,
                      batch_column=batch_column or None, sample_type=sample_type, crystal=atr_crystal,
                      mode=_mode, output_dir=_out, dataset_name=dataset_name)
_report = open(os.path.join(result["output_dir"], "report.txt"), encoding="utf-8").read()
print("\n" + "=" * 100 + "\n" + _report.split("APPENDIX")[0])
files.download(result["zip"])

## How to read the results

- **SUMMARY** (top of `report.txt`): the verdict. If *models restricted to regions without sample bands* classify well above chance (≥ 0.65), the groups differ in acquisition conditions and no classification result can be interpreted as sample biochemistry. The same applies when class and batch/site are strongly associated.
- **fig6_acquisition_audit.png**: the negative controls (hatched bars) should stay near 0.5.
- **Variables with the largest differences**: what changes between groups, in chemical terms (glossary at the end of the report and in `variable_glossary.csv`).
- **Audit** mode compares the full spectrum, the descriptor and the controls; **Full** mode adds all reference and compact methods, non-inferiority tests, robustness to perturbations and permutation tests (used for publication).

Bands of sample-type modules (urine, salivary thiocyanate) are based on the literature and have **not** been validated on data; the report flags them.

**Please cite** the DANTE-IR article and software DOI when you use it.